$${\color{yellow}{\text{Deep Learning Webinar-2, Saturday, May 3, 2025}}}$$



---

Load essential libraries

---

In [ ]:
!pip install --upgrade numpy
!pip install --upgrade gensim

---

After reinstalling numpy and gensim, go to *Runtime -> Restart Session* and execute from the following cell onward:

---

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
plt.style.use('dark_background')
%matplotlib inline
import sys
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
import nltk
import gensim.downloader
from nltk.tokenize import word_tokenize

---

Mount Google Drive folder if running Google Colab

---

In [ ]:
## Mount Google drive folder if running in Colab
if('google.colab' in sys.modules):
    from google.colab import drive
    drive.mount('/content/drive', force_remount = True)
    DIR = '/content/drive/MyDrive/Colab Notebooks/MAHE/Office of Online Education/MDS6304_Webinar_April2025'
    DATA_DIR = DIR+'/Data/'
else:
    DATA_DIR = 'Data/'

Mounted at /content/drive


---

**We will now use Pytorch to create tensors**

The patient data matrix:

![patient data matrix](https://1drv.ms/i/s!AjTcbXuSD3I3hsxIkL4V93-CGq8RkQ?embed=1&width=1000)

**Notation**:

Zeroth patient vector $\mathbf{x}^{(0)}= \begin{bmatrix}72\\120\\37.3\\104\\32.5\end{bmatrix}$ and zeroth feature (heart rate vector) $\mathbf{x}_0 = \begin{bmatrix}72\\85\\68\\90\\84\\78\end{bmatrix}.$

---



In [ ]:
## Create a patient data matrix as a constant tensor
X = torch.tensor([[72, 120, 37.3, 104, 32.5],
                 [85, 130, 37.0, 110, 14],
                 [68, 110, 38.5, 125, 34],
                 [90, 140, 38.0, 130, 26],
                 [84, 132, 38.3, 146, 30],
                 [78, 128, 37.2, 102, 12]])
print(type(X))
print(X)
print(X.shape)
# X is an rank-2 tensor which is similar to a numpy 2D array
print(X[0]) # this is patient-0 info which is a rank-1 tensor
print(X[0, 2]) # the 2nd feature of the 0th patient (temperature of the 0th patient)
print(X.dtype)

---

**Convert a PyTorch object into a numpy array**

---

In [ ]:
X_numpy = X.numpy()
print(type(X_numpy))
print(X_numpy)
print(X_numpy.shape)

---

**Addition and subtraction of vectors, scalar multiplication (apply operation componentwise)**

![vector addition](https://1drv.ms/i/c/37720f927b6ddc34/IQQ03G17kg9yIIA3NokBAAAAAZLAaAoWwhtn8Vk26NotALo?width=256)

![vector subtracton](https://1drv.ms/i/c/37720f927b6ddc34/IQQ03G17kg9yIIA3M4kBAAAAAU_n_mAEv006QFZm_sUj2Dc?width=256)

![vector multiplication](https://1drv.ms/i/c/37720f927b6ddc34/IQQ03G17kg9yIIA3NIkBAAAAAa_qL04bLT4kWoNeHcrR9LQ?width=256)

![vector geometry1](https://1drv.ms/i/c/37720f927b6ddc34/IQSGNMr5z3SSRry7LSKL7LybAcGYuzgw5smabV8-6DudXIs?width=230)

![vector geometry2](https://1drv.ms/i/c/37720f927b6ddc34/IQQ03G17kg9yIIA3WokBAAAAAQi8FPV9YCebl5WnyEKJ3vg?width=213&height=192)


---

In [ ]:
# Vector addition
print(X[1, :] + X[2, :])

# Vector subtraction
print(X[1, :] - X[2, :])

# Scalar-vector multiplication
print(X[:, 2]) # the temperature of all patients
# Operation not defined in pen & paper but in computation is referred to as
# broadcasting
print((9/5)*X[:, 2] + 32)

# Average patient
print((1/6) * (X[0, :] + X[1, :] + X[2, :] + X[3, :] + X[4, :] + X[5, :]))
print(torch.mean(X, dim = 0)) # dim = 0 means top-to-bottom

---

Application of vector subtraction in natural language processing (NLP): download the word embedding model trained on Wikipedia articles.

---

In [ ]:
model = gensim.downloader.load('glove-wiki-gigaword-50')

[==================================================] 100.0% 66.0/66.0MB downloaded


---

Now we will see what embedding vector comes as a result of applying the model for the words *cricket* and *football*.

Next, we will do an *intuitive* subtraction of word embeddings as in

1. Cricket without Tendulkar
2. Football without Messi

Note that the embedding vectors have 50 components corresponding to the 50-dimensional embedding of model suggested by the name '**glove-wiki-gigaword-50**'

---

In [ ]:
print(model['cricket'])
print(model['football'])
a = model['cricket'] - model['tendulkar'] # cricket without tendulkar
b = model['football'] - model['messi'] # football without messi
print(a)
print(b)

[-1.2233    -0.28292   -1.7429     0.56621   -0.13787   -0.88088
 -0.26887    0.41893    0.94977   -0.61332    0.0092522  1.0045
 -0.89004   -0.55148    0.61202    0.42256    0.92935    0.83307
 -1.5568     0.084345  -0.018224   0.84803    0.68321    0.97059
  0.26561   -1.0538     0.40724   -0.45079   -0.89013    0.94153
  2.2055     0.75363    0.5166     0.47977    0.86824    0.57228
  0.81846   -0.070934  -0.9391    -0.81628   -0.35486   -0.010408
 -0.83316    1.1001    -0.087408   1.8452    -0.83112    0.43732
  0.63007   -0.81023  ]
[-1.8209    0.70094  -1.1403    0.34363  -0.42266  -0.92479  -1.3942
  0.28512  -0.78416  -0.52579   0.89627   0.35899  -0.80087  -0.34636
  1.0854   -0.087046  0.63411   1.1429   -1.6264    0.41326  -1.1283
 -0.16645   0.17424   0.99585  -0.81838  -1.7724    0.078281  0.13382
 -0.59779  -0.45068   2.5474    1.0693   -0.27017  -0.75646   0.24757
  1.0261    0.11329   0.17668  -0.23257  -1.1561   -0.10665  -0.25377
 -0.65102   0.32393  -0.58262   0.8813

---

A tensor of rank 3 corresponding to 4 time stamps (hourly), 3 samples (patients), 2 features (HR and BP). Assume that admission time is 9AM.

---

In [ ]:
# A rank-3 patient tensor with shape (4, 3, 2)
# with meaning for
# dim-0 as 4 hourly timestamps,
# dim-1 as 3 patients, and
# dim-2 as 2 features (HR and BP)
T = torch.tensor([[[74., 128], [79, 116], [71, 116]],
                 [[78, 118], [82, 124], [72, 128]],
                 [[84, 138], [84, 130], [74, 120]],
                 [[82, 126], [76, 156], [82, 132]]])
print(T)

tensor([[[ 74., 128.],
         [ 79., 116.],
         [ 71., 116.]],

        [[ 78., 118.],
         [ 82., 124.],
         [ 72., 128.]],

        [[ 84., 138.],
         [ 84., 130.],
         [ 74., 120.]],

        [[ 82., 126.],
         [ 76., 156.],
         [ 82., 132.]]])


In [ ]:
print(T[:, 0, :])
print(torch.unsqueeze(T[:, 0, :], 1))

tensor([[ 74., 128.],
        [ 78., 118.],
        [ 84., 138.],
        [ 82., 126.]])
tensor([[[ 74., 128.]],

        [[ 78., 118.]],

        [[ 84., 138.]],

        [[ 82., 126.]]])


---

**Accessing elements of a tensor**

---

In [ ]:
## Accessing elements of a tensor
# Rank-3 tensor T has axes order (timestamps, patients, features)

# Element of T at postion 3 w.r.t. axis-0, position 2 w.r.t. axis-1,
# position-1 w.r.t axis-2
print(T[3, 2, 1]) # BP of patient-2 at noon

# Element-0 of object T which is also the info for all patients at
# admission time 9AM
print(T[0])

# Patient-2 info at 12PM
print(T[3, 2])

tensor(132.)
tensor([[ 74., 128.],
        [ 79., 116.],
        [ 71., 116.]])
tensor([ 82., 132.])


---

**Broadcasting**

---

In [ ]:
# A simple broadcasting example
a = torch.tensor([[1.0, 2.0, 3.0]])
b = torch.tensor([4.0])
print(a.shape)
print(b.shape)
print(a-b)

torch.Size([1, 3])
torch.Size([1])
tensor([[-3., -2., -1.]])


In [ ]:
# How to add a new axis to a tensor using the unqueeze() method
print(T.shape)
print(T[:, 0, :].shape)
print(torch.unsqueeze(T[:, 0, :], 1).shape)
T - torch.unsqueeze(T[:, 0, :], 1)

torch.Size([4, 3, 2])
torch.Size([4, 2])
torch.Size([4, 1, 2])


tensor([[[  0.,   0.],
         [  5., -12.],
         [ -3., -12.]],

        [[  0.,   0.],
         [  4.,   6.],
         [ -6.,  10.]],

        [[  0.,   0.],
         [  0.,  -8.],
         [-10., -18.]],

        [[  0.,   0.],
         [ -6.,  30.],
         [  0.,   6.]]])

In [ ]:
# How different are the patients from patient-0?
#T - T[:, 0, :] # does not work for broadcasting
print(T - torch.unsqueeze(T[:, 0, :], 1))

#  # How different are the patients compared to their time at admission
print(T - T[0])

tensor([[[  0.,   0.],
         [  5., -12.],
         [ -3., -12.]],

        [[  0.,   0.],
         [  4.,   6.],
         [ -6.,  10.]],

        [[  0.,   0.],
         [  0.,  -8.],
         [-10., -18.]],

        [[  0.,   0.],
         [ -6.,  30.],
         [  0.,   6.]]])
tensor([[[  0.,   0.],
         [  0.,   0.],
         [  0.,   0.]],

        [[  4., -10.],
         [  3.,   8.],
         [  1.,  12.]],

        [[ 10.,  10.],
         [  5.,  14.],
         [  3.,   4.]],

        [[  8.,  -2.],
         [ -3.,  40.],
         [ 11.,  16.]]])


---

**Exercise**: interpret $\texttt{T[:, -1, :]}$

---

In [ ]:
# Last patient's info at all timestamps
T[:, -1, :]

tensor([[ 71., 116.],
        [ 72., 128.],
        [ 74., 120.],
        [ 82., 132.]])

---

$l_2$ norm or the geometric length of a vector denoted as $\lVert \mathbf{a}\rVert$ tells us how long a vector is. In 2-dimensions, $$\lVert \mathbf{a}\rVert_2 = \sqrt{a_1^2+a_2^2}$$ and in $n$-dimensions, $$\lVert \mathbf{a}\rVert_2 = \sqrt{a_1^2+a_2^2+\cdots+a_n^2}.$$

![vector norm](https://1drv.ms/i/c/37720f927b6ddc34/IQT817WmpQjlRqZ1R0d5Cfv6AUW6c4robL-gk06i9wmCaFU?width=500)

---

In [ ]:
## l2 norm of a vector
x = torch.tensor([76.0, 124.0])
print(x)
print(torch.norm(x))

tensor([ 76., 124.])
tensor(145.4373)



---

**Dot Product of Vectors**

A scalar resulting from an elementwise multiplication and addition: $$\mathbf{a}{\color{cyan}\cdot}\mathbf{b} = {\color{red}{a_1b_1}}+{\color{green}{a_2b_2}}+\cdots+{\color{magenta}{a_nb_n}}$$

The <font color="cyan">dot</font> ${\color{cyan}\cdot}$ represents the computation of the dot product.


---

In [ ]:
## Dot product of vectors
a = torch.tensor([1.0, 2, 3])
b = torch.tensor([4.0, 5, 6])
torch.dot(a, b)

tensor(32.)

---

The dot product is a measure of similarity between vectors (or, how aligned they are geometrically).

![dot product](https://1drv.ms/i/c/37720f927b6ddc34/IQTbcGSjdbhSTJ7J39d5BCWAAWS6-y5U6J87vHuDWeAqGwM?width=6000)
---

In [ ]:
a = torch.tensor([1.0, 2.0])
b = torch.tensor([2.0, 4.0])
c = torch.tensor([-2.0, 1.0])
d = torch.tensor([-1.0, -2.0])
print(torch.dot(a, b))
print(torch.dot(a, c))
print(torch.dot(a, d))

tensor(10.)
tensor(0.)
tensor(-5.)


---

Cauchy-Schwarz inequality $-1\leq\frac{\mathbf{x}\cdot{\mathbf{y}}}{\lVert\mathbf{x}\rVert_2\lVert\mathbf{y}\rVert_2}\leq1.$

This is a normalized measure of similarity (or extent of alignment) between vectors.

Angle between vectors $\mathbf{x}$ and $\mathbf{y} = \cos^{-1}\left(\frac{\mathbf{x}\cdot{\mathbf{y}}}{\lVert\mathbf{x}\rVert_2\lVert\mathbf{y}\rVert_2}\right).$

![angle](https://1drv.ms/i/c/37720f927b6ddc34/IQQ03G17kg9yIIA3WokBAAAAAQi8FPV9YCebl5WnyEKJ3vg?width=213&height=400)


---

In [ ]:
x = torch.tensor([1.0, 2.0])
y = torch.tensor([2.0, 1.0])
print(torch.norm(x - y))
# Angle between x and y in radians
print(torch.acos(torch.dot(x, y) / (torch.norm(x) * torch.norm(y))))
# Angle between x and y in degrees
print((180.0/torch.pi)*torch.acos(torch.dot(x, y) / (torch.norm(x) * torch.norm(y))))

tensor(1.4142)
tensor(0.6435)
tensor(36.8699)


---

Application of the Cauchy-Schwarz inequality: is "Cricket without Tendulkar" same as "Football without Messi"?

---

In [ ]:
a = model['cricket'] - model['tendulkar']
b = model['football'] - model['messi']
#print(np.dot(a, b)/(np.linalg.norm(a)*np.linalg.norm(b))) # normalized similarity
print((180/np.pi)*np.arccos(np.dot(a, b)/(np.linalg.norm(a)*np.linalg.norm(b)))) # angular difference in degrees
print(np.linalg.norm(a-b)) # linear difference

42.51263077162803
4.2349043



---

**Hadamard Product of Vectors**

A vector resulting from an elementwise multiplication: $$\mathbf{a}{\color{cyan}\otimes}\mathbf{b} = \begin{bmatrix}{\color{red}{a_1\times b_1}}\\{\color{green}{a_2\times b_2}}\\\vdots\\{\color{magenta}{a_n\times b_n}}\end{bmatrix}.$$

The <font color="cyan">$\otimes$</font> represents the computation of the Hadamard product.

---

In [ ]:
## Hadamard product
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

# Element-wise multiplication (Hadamard product)
print(a * b) # using the * operator
print(torch.mul(a, b)) # using the torch.mul() function

tensor([ 4., 10., 18.])
tensor([ 4., 10., 18.])


---

A matrix-vector product is simply a sequence of dot products of the rows of the matrix (seen as vectors) with the vector

![matvec product](https://1drv.ms/i/c/37720f927b6ddc34/IQQ1cQ8fZdFmS4cnGkBlsZbAAaL2zMtzWdjHe-HCMt4UTA0?width=700)

---

In [ ]:
## Matrix-vector product
A = torch.tensor([[1.0, 2.0, 4.0],
                  [2.0, -1.0, 3.0]])
x = torch.tensor([4.0, 2.0, -2.0])

# Matrix-vector multiplication
print(A)
print(x)
torch.matmul(A, x)

tensor([[ 1.,  2.,  4.],
        [ 2., -1.,  3.]])
tensor([ 4.,  2., -2.])


tensor([0., 0.])

---

Here we create a simple sentence in English and tokenize it

---

In [ ]:
sentence = 'i swam quickly across the river to get to the other bank'
nltk.download('punkt_tab')
tokens = word_tokenize(sentence)
print(len(tokens))
print(tokens)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...


12
['i', 'swam', 'quickly', 'across', 'the', 'river', 'to', 'get', 'to', 'the', 'other', 'bank']


[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


---

Generate the word embeddings for the tokens and store them in a matrix $\mathbf{X}$ such that each row of the matrix corresponds to a token.

---

In [ ]:
X_word = torch.tensor(model[tokens]) # Word-embedding matrix
print(X_word.shape)
print(X_word[1])

torch.Size([12, 50])
tensor([-0.3530,  0.3695,  0.4727,  0.0138, -0.1648, -0.5269, -0.7399,  1.2058,
         1.1147, -0.4677,  0.1793, -0.5924,  0.2526,  0.4345,  0.6702, -0.2859,
         0.7311,  0.3183, -1.5825, -0.3571, -0.1235,  1.3207,  0.9005,  0.1220,
         1.4916, -0.5411,  1.2264,  0.3394, -0.3305, -0.3546, -0.0961,  0.0503,
         0.1248, -0.0228, -0.3539, -0.4319,  0.0371,  0.7144, -0.0158, -0.4602,
        -0.3809, -0.6541,  1.2373,  0.7776,  0.0636, -1.4376,  0.2417, -1.5705,
        -0.1064, -0.8271])


---

A matrix-matrix product is simply a sequence of matrix-vector products.

![matmatprod](https://1drv.ms/i/c/37720f927b6ddc34/IQQ-B3z7tbWHQqBrW9k2ElDVAUc5fWzM24txLkgBK7f8Yac?width=550)


---

In [ ]:
## Matrix-matrix product
A = torch.tensor([[1.0, 2.0, 4.0],
                  [2.0, -1.0, 3.0]])
B = torch.tensor([[4.0, -1.0],
                  [2.0, 0.0],
                  [-2.0, 3.0]])
torch.matmul(A, B)

tensor([[ 0., 11.],
        [ 0.,  7.]])

---

The similarity between each pair of words represented in the word embeddings matrix $\mathbf{X}_\mathrm{word}$ is the matrix-matrix product $\mathbf{X}_\mathrm{word}\mathbf{X}_\mathrm{word}^\mathrm{T}.$

---

In [ ]:
S = torch.matmul(X_word, X_word.T)
print(S)
print(S.shape)

tensor([[39.0197,  4.5574, 20.2995, 14.5481, 19.4420, 12.0143, 22.6571, 29.8528,
         22.6571, 19.4420, 17.8858, 11.2500],
        [ 4.5574, 24.7064,  4.2668,  7.2292,  0.9357,  8.8437,  2.3160,  5.4718,
          2.3160,  0.9357, -0.6112, -1.5239],
        [20.2995,  4.2668, 21.2841, 16.5796, 16.1261, 12.0684, 18.8431, 19.4523,
         18.8431, 16.1261, 15.2439, 12.9733],
        [14.5481,  7.2292, 16.5796, 28.8804, 19.3706, 22.9543, 17.9298, 16.6827,
         17.9298, 19.3706, 19.6938, 14.1935],
        [19.4420,  0.9357, 16.1261, 19.3706, 24.6793, 18.2549, 21.1141, 17.5502,
         21.1141, 24.6793, 19.8692, 18.0800],
        [12.0143,  8.8437, 12.0684, 22.9543, 18.2549, 44.2210, 14.7193, 12.0669,
         14.7193, 18.2549, 12.5996, 16.6162],
        [22.6571,  2.3160, 18.8431, 17.9298, 21.1141, 14.7193, 24.5706, 22.2478,
         24.5706, 21.1141, 20.6623, 19.2235],
        [29.8528,  5.4718, 19.4523, 16.6827, 17.5502, 12.0669, 22.2478, 30.1930,
         22.2478, 17.5502, 19.